# Initializing kcat values for PAMparametrizer

In this notebook, we give an overview of how an initial set of turnover number can be obtained. To enable this, we use multiple databases to map kcat values obtained from machine learning (from the [GotEnzymes](https://metabolicatlas.org/gotenzymes) database) to the proteins and reactions in the model using the gene-protein-reaction associations.

Before we start, we have to set up all the package and files required. Adjust the filenames and paths as desired. Importantly, make sure to **adjust the regular expression in the cell below** in order to find the gene names associated with the organism you want to model.

In [1]:
import pandas as pd
import os
from cobra.io import load_json_model, read_sbml_model
import matplotlib.pyplot as plt
import re
import numpy as np
import datetime

if os.path.split(os.getcwd())[1]=='i1_preprocessing':
    os.chdir('../..')
    
from PAModelpy.utils.pam_generation import merge_enzyme_complexes, get_protein_gene_mapping, _parse_gpr


DEFAULT_PROT_MW = 39959.4825 #g/mol
DEFAULT_KCAT =  13.7 #/s median value from BRENDA

PAM_DATA_FILE_PATH = os.path.join('Data', 'proteinAllocationModel_iML1515_EnzymaticData_new.xls')
UNIPROT_FILE_PATH = os.path.join('Data', 'Databases','uniprotkb_cglutamicumATCC13032_250124.xlsx')
NCBI_FILE_PATH = os.path.join('Data','Cglutanicum_phenotypes','NCBI_genes_corynglutatcc13032_250207.tsv')
GOTENZYMES_FILE_PATH = os.path.join('Data','Databases', 'GotEnzymes_cgl_250124.json')

Loading PAModelpy modules version 0.0.4.6


In [2]:
#output file path
# Create a datetime object for the current date
current_date = datetime.datetime.now()
# Format the date as yymmdd
formatted_date = current_date.strftime('%y%m%d')

OUTPUT_FILE_PATH = os.path.join('Results', '1_preprocessing', f'proteinAllocationModel_iCGB21FR_EnzymaticData_{formatted_date}.xlsx')
AE_OUTPUT_FILE_PATH = os.path.join('Results', '1_preprocessing', f'{formatted_date}_cgl_ActiveEnzymeInformation_GotEnzymes.xlsx')

Relatively complicated regex because of complex gene names

1. WP_\d+_\d+: Matches identifiers starting with WP_ followed by digits (\d+), an underscore, and more digits.
2. |: Alternation operator to separate different patterns.
3. lcl_NC_\d+_\d+_prot_: Matches strings starting with lcl_NC_, followed by digits, an underscore, more digits, _prot_, and then:

    - WP_\d+_\d+_\d+: Matches WP_ identifiers with an additional _ and digits at the end.
    - |\d+: Matches a plain numeric sequence following _prot_.

4. |: Alternation operator for the next part.
5. cgl\d{4}: Matches strings starting with cgl followed by exactly 4 digits.

In [3]:
#You need to adjust this to find the geneid/locustag for your microbe
locustag_regex =r'(?:WP_\d+_\d+|lcl_NC_\d+_\d+_prot_(?:WP_\d+_\d+_\d+|\d+)|[cC]gl\d{4}|[cC]g\d{4})'

## 0. Download information from different databases
1. **Metabolic model stoichiometries and reactions: [BioModels](https://www.ebi.ac.uk/biomodels/)**
- Get your model! Or use your own reaction identifiers
- In this example we will use the iCGB21FR model for [*Corynebacterium glutamicum* ATCC 13032](https://www.ebi.ac.uk/biomodels/MODEL2102050001#Overview)

2. **Turnover numbers: [GotEnzymes](https://metabolicatlas.org/gotenzymes)**
- Go to the [API of the Metabolic Atlas](https://metabolicatlas.org/api/v2/#/GotEnzymes/gotEnzymes)
- Use the `GET\gotenzymes\enzymes?` functionality, click `Try it out`
- Here we filled in `cgl` (*C. glutanicum* ATCC 13032) for `organism` and `10000` for `pagination[pageSize]`. The rest was left empty
- Click execute and a json file with all the information will be downloaded

3. **Gene-Protein-Reaction relations: [UniprotKB](https://www.uniprot.org/)**
- Go to the [Uniprot knowledge base](https://www.uniprot.org/) and used advanced search to query for your organism. For *C. glutanicum* we used [this](https://www.uniprot.org/uniprotkb?query=(taxonomy_id:196627)) webpage
- Click download, select `format`:`Excel` and select the following: 
    - `Uniprot Data -> Names & Taxonomy`: `Gene Names`
    - `Uniprot Data -> Sequences`: `length`, `mass`
    - `Uniprot Data -> Function`: `RheaID`
- Download the resulting Excel file

4. **NCBI gene id to KEGG gene id mapping**: [NCBI database](https://www.ncbi.nlm.nih.gov/)
- The model identifiers for this model are in NCBI format, but we need the KEGG locus tag ids to map the genes to the corresponding Uniprot ids. Therefore, we use the mapping provided by NCBI.
- Go to the NCBI gene list for *C. glutanicum* on [NCBI genes](https://www.ncbi.nlm.nih.gov/gene/?term=Corynebacterium+glutamicum+ATCC+13032)
- On the top left click `send to`, select `file` and `text`

## 1. Get information from the model

In the metabolic model itself is already quite some information stored. In order to obtain up-to-date information on the enzymes related to the reaction, we will need to obtain the kegg reaction ID, the reactants, the product and the gene-protein relation.

In [4]:
def create_id_mapper_from_model(model):
    expand=False

    #make a id mapping and create a mapping dataframe
    id_mapper_df = pd.DataFrame(columns = ['rxn_id', 'kegg_id', 'rhea_id', 'mnx_id','Reactants','Products','EC', 'GPR'])
    for rxn in model.reactions:
        #skip transport reactions and ATP maintenance requirement
        if 'ex' in rxn.id or 'EX' in rxn.id or rxn.id == 'ATPM':
            continue
        to_append= []   

        to_append.append(rxn.id)
        #not all reactions are annotated with a KEGG ID in the model
        try: 
            to_append.append(rxn.annotation['kegg.reaction'])
        except:
            to_append.append(np.nan)
        try: 
            to_append.append(rxn.annotation['rhea'])
        except:
            to_append.append(np.nan)
        try: 
            to_append.append(rxn.annotation['metanetx.reaction'])
        except:
            to_append.append(np.nan)
        try:
            to_append.append([reac.annotation['kegg.compound'] for reac in rxn.reactants])
        except:
            to_append.append([])
        try: 
            to_append.append([prod.annotation['kegg.compound'] for prod in rxn.products])
        except:
            to_append.append([])
        if 'ec-code' in rxn.annotation.keys():
            #if there are multiple enzymes make seperate lists
            if isinstance(rxn.annotation['ec-code'], list) and expand:
                info = to_append.copy()
                for i in range(len(rxn.annotation['ec-code'])):
                    #make a new row for each enzyme by copying the previous information
                    info = to_append.copy()
                    ec= rxn.annotation['ec-code'][i]
                    info.append(ec)
                    #add gpr
                    info.append(rxn.gene_reaction_rule)
                    #add to df
                    id_mapper_df.loc[len(id_mapper_df)] = info
            else:
                to_append.append(rxn.annotation['ec-code'])
                to_append.append(rxn.gene_reaction_rule)
                id_mapper_df.loc[len(id_mapper_df)] = to_append
        else:
            to_append.append(np.nan)
            #ignore entries without enzyme and gpr relation
            if rxn.gene_reaction_rule =='':
                continue
            else:
                to_append.append(rxn.gene_reaction_rule)
            id_mapper_df.loc[len(id_mapper_df)] = to_append

    #remove entries without GRP relation or Ec number


    print('Number of reactions in the mapping dataframe: ',len(id_mapper_df))
    print('Number of reactions mapped to KEGG identifier: ', len(id_mapper_df.kegg_id.dropna()))
    print('Number of reactions mapped to rhea identifier: ', len(id_mapper_df.rhea_id.dropna()))
    print('Number of reactions mapped to MetaNetX identifier: ', len(id_mapper_df.mnx_id.dropna()))
    return id_mapper_df

print('mapping the ids from the iCW773 model')
id_mapper_icw = create_id_mapper_from_model(load_json_model(os.path.join('Models', 'iCW773_Niu2022.json')))

print('\nmapping the ids from the iCGB21FR model')
model =read_sbml_model(os.path.join('Models', 'iCGB21FR.xml'))
id_mapper_icg = create_id_mapper_from_model(model)


mapping the ids from the iCW773 model
Set parameter Username
Academic license - for non-commercial use only - expires 2025-03-06


SBML package 'layout' not supported by cobrapy, information is not parsed
https://juser.fz-juelich.de/record/188973 does not conform to 'http(s)://identifiers.org/collection/id' or'http(s)://identifiers.org/COLLECTION:id


Number of reactions in the mapping dataframe:  804
Number of reactions mapped to KEGG identifier:  0
Number of reactions mapped to rhea identifier:  0
Number of reactions mapped to MetaNetX identifier:  0

mapping the ids from the iCGB21FR model
Number of reactions in the mapping dataframe:  1147
Number of reactions mapped to KEGG identifier:  491
Number of reactions mapped to rhea identifier:  643
Number of reactions mapped to MetaNetX identifier:  1120


In [5]:
id_mapper_df = pd.merge(id_mapper_icg, id_mapper_icw[['rxn_id', 'GPR']], on='rxn_id', how = 'left')
id_mapper_df

,rxn_id,kegg_id,rhea_id,mnx_id,Reactants,Products,EC,GPR_x,GPR_y
0,2AGPEAT160,NaN,NaN,MNXR94760,[],[],2.3.1.40,,NaN
1,2AGPEAT180,NaN,NaN,MNXR94762,[],[],2.3.1.40,,NaN
2,2MAHMP,NaN,NaN,MNXR142616,"[[C00001, C01328, C00001;C01328;D00001;D03703;...","[C04556, C00080, [C00009, C00009;D05467]]",NaN,lcl_NC_006958_1_prot_WP_003855288_1_2949 or lc...,Cgl1176
3,3HAD140,R04568,NaN,MNXR94879,[C04688],"[[C00001, C01328, C00001;C01328;D00001;D03703;...","[4.2.1.61, 2.3.1.86, 2.3.1.-, 2.3.1.85, 4.2.1.59]",,Cgl0289
4,3MBt2pp,NaN,NaN,MNXR94916,"[C00080, C08262]","[C08262, C00080]",NaN,lcl_NC_006958_1_prot_WP_011013917_1_810,NaN
...,...,...,...,...,...,...,...,...,...
1142,Ca2t2,NaN,NaN,NaN,"[C00076, C00080]","[C00076, C00080]",NaN,WP_011014107_1,NaN
1143,CAT,R00009,"[20309, 20310, 20311, 20312]",MNXR96455,"[[C00027, C00027;D00008]]","[[C00001, C01328, C00001;C01328;D00001;D03703;...","[1.11.1.21, 1.11.1.6]",WP_011013509_1,Cgl0255
1144,SUCDi,NaN,"[29187, 29188, 29189, 29190]",MNXR99641,[],[],NaN,WP_011013606_1 and WP_011013607_1,
1145,PRFGS_1,NaN,"[17129, 17131, 17130, 17132]",MNXR138721,"[[C00002, C00002;D08646], C04376, [C00064, C00...","[[C00008, C00008;G11113], C04640, [C00025, C00...",6.3.5.3,lcl_NC_006958_1_prot_WP_003854010_1_2470,NaN


### Map the NCBI gene names to KEGG gene names

In the iCGB21FR model, the gene names are by default the ncbi gene names. To map the gene-protein-reaction associations to the corresponding uniprot peptide identifiers, we need to map the model gene names to the kegg gene names.

In [6]:
ncbi_genes = pd.read_csv(NCBI_FILE_PATH,sep = '\t')
ncbi_genes.head()

,tax_id,Org_name,GeneID,CurrentID,Status,Symbol,Aliases,description,other_designations,map_location,chromosome,genomic_nucleotide_accession.version,start_position_on_the_genomic_accession,end_position_on_the_genomic_accession,orientation,exon_count,OMIM,Unnamed: 17
0,196627,Corynebacterium glutamicum ATCC 13032,1019166,0,live,CGL_RS05895,"CGL_RS05895, NCgl1136",homoserine dehydrogenase,homoserine dehydrogenase,NaN,NaN,NC_003450.3,1242507.0,1243844.0,plus,0,NaN,NaN
1,196627,Corynebacterium glutamicum ATCC 13032,1019114,0,live,CGL_RS05635,"CGL_RS05635, NCgl1084",multifunctional oxoglutarate decarboxylase/oxo...,multifunctional oxoglutarate decarboxylase/oxo...,NaN,NaN,NC_003450.3,1172498.0,1176163.0,minus,0,NaN,NaN
2,196627,Corynebacterium glutamicum ATCC 13032,1020564,0,live,CGL_RS13060,"CGL_RS13060, NCgl2528",diaminopimelate dehydrogenase,diaminopimelate dehydrogenase,NaN,NaN,NC_003450.3,2786754.0,2787716.0,minus,0,NaN,NaN
3,196627,Corynebacterium glutamicum ATCC 13032,92792280,0,live,trpL,CGL_RS15555,trp operon leader peptide,trp operon leader peptide,NaN,NaN,NC_003450.3,3233152.0,3233205.0,plus,0,NaN,NaN
4,196627,Corynebacterium glutamicum ATCC 13032,1020138,0,live,pimB,"CGL_RS10855, NCgl2106",GDP-mannose-dependent monoacylated alpha-(1-6)...,GDP-mannose-dependent monoacylated alpha-(1-6)...,NaN,NaN,NC_003450.3,2317631.0,2318776.0,minus,0,NaN,NaN


In [ ]:
from Bio import SeqIO

# Replace with the path to your GPFF file or URL to download it
gpff_file_bielfeld = "Data/Cglutanicum_phenotypes/GCA_000196335.1_ASM19633v1_protein.gpff"  # Use a local file or an online URL
gpff_file_kyoto = "Data/Cglutanicum_phenotypes/GCF_000011325.1_ASM1132v1_protein.gpff"  # Use a local file or an online URL

bielefeld_records = SeqIO.parse(gpff_file_bielfeld, "genbank")
kyoto_records = SeqIO.parse(gpff_file_kyoto, "genbank")

# Parse the GPFF file
i=0

from collections import defaultdict

mapping = defaultdict(list)

while i<10:
    for record in bielefeld_records:
        # Loop over the features to find db_xrefs
        for feature in record.features:
            if "db_xref" in feature.qualifiers:
                reference = dict()
                for db_xref in feature.qualifiers["db_xref"]:
                    ref = db_xref.strip().split(':')
                    reference[ref[0]] = ref[1]
                for rec in kyoto_records:
                    if record.seq == rec.seq:
                        mapping[rec.id].append(reference)
            
    
print(mapping)

{'taxon': '196627'}
{'GOA': 'Q8NUD8', 'InterPro': 'IPR027417'}
{'taxon': '196627'}
{'UniProtKB/TrEMBL': 'Q8NUD7'}
{'taxon': '196627'}
{'GOA': 'Q8NUD6', 'InterPro': 'IPR022637', 'UniProtKB/TrEMBL': 'Q8NUD6'}
{'taxon': '196627'}
{'GOA': 'Q6M8X7', 'InterPro': 'IPR027417'}
{'taxon': '196627'}
{'InterPro': 'IPR023007'}
{'taxon': '196627'}
{'GOA': 'Q6M8X6', 'InterPro': 'IPR020568', 'UniProtKB/TrEMBL': 'Q6M8X6'}
{'taxon': '196627'}
{'InterPro': 'IPR029058', 'UniProtKB/TrEMBL': 'Q8NUD2'}
{'taxon': '196627'}
{'InterPro': 'IPR003740', 'UniProtKB/TrEMBL': 'Q8NUD1'}
{'taxon': '196627'}
{'UniProtKB/TrEMBL': 'Q8NUD0'}
{'taxon': '196627'}
{'InterPro': 'IPR011991', 'UniProtKB/TrEMBL': 'Q8NUC9'}
{'taxon': '196627'}
{'UniProtKB/TrEMBL': 'Q8NUC8'}
{'taxon': '196627'}
{'UniProtKB/TrEMBL': 'Q8NUC7'}
{'taxon': '196627'}
{'GOA': 'Q8NUC6', 'InterPro': 'IPR024946', 'UniProtKB/TrEMBL': 'Q8NUC6'}
{'taxon': '196627'}
{'InterPro': 'IPR021949', 'UniProtKB/TrEMBL': 'Q8NUC5'}
{'taxon': '196627'}
{'GOA': 'Q6M8W7', 'In

{'taxon': '196627'}
{'InterPro': 'IPR029068', 'UniProtKB/TrEMBL': 'Q8NRH4'}
{'taxon': '196627'}
{'GOA': 'Q8NRH3', 'InterPro': 'IPR020846', 'UniProtKB/TrEMBL': 'Q8NRH3'}
{'taxon': '196627'}
{'GOA': 'Q6M689', 'InterPro': 'IPR016040', 'UniProtKB/TrEMBL': 'Q6M689'}
{'taxon': '196627'}
{'InterPro': 'IPR017195', 'UniProtKB/TrEMBL': 'Q6M688'}
{'taxon': '196627'}
{'GOA': 'Q8NRH0', 'InterPro': 'IPR027417', 'UniProtKB/TrEMBL': 'Q8NRH0'}
{'taxon': '196627'}
{'InterPro': 'IPR003339', 'UniProtKB/TrEMBL': 'Q8NRG9'}
{'taxon': '196627'}
{'GOA': 'Q8NRG8', 'InterPro': 'IPR027417', 'UniProtKB/TrEMBL': 'Q8NRG8'}
{'taxon': '196627'}
{'GOA': 'Q8NRG7', 'InterPro': 'IPR004837', 'UniProtKB/TrEMBL': 'Q8NRG7'}
{'taxon': '196627'}
{'InterPro': 'IPR024078', 'UniProtKB/TrEMBL': 'Q6M683'}
{'taxon': '196627'}
{'InterPro': 'IPR011737', 'UniProtKB/TrEMBL': 'Q8NRG5'}
{'taxon': '196627'}
{'GOA': 'Q8NRG4', 'InterPro': 'IPR027417', 'UniProtKB/TrEMBL': 'Q8NRG4'}
{'taxon': '196627'}
{'GOA': 'Q8NRG3', 'InterPro': 'IPR018219'}

{'taxon': '196627'}
{'GOA': 'Q6M2J6', 'InterPro': 'IPR006680', 'UniProtKB/TrEMBL': 'Q6M2J6'}
{'taxon': '196627'}
{'GOA': 'Q8NMD2', 'InterPro': 'IPR013785', 'UniProtKB/TrEMBL': 'Q8NMD2'}
{'taxon': '196627'}
{'GOA': 'Q8NMD1', 'InterPro': 'IPR000600', 'UniProtKB/TrEMBL': 'Q8NMD1'}
{'taxon': '196627'}
{'GOA': 'Q8NMD0', 'InterPro': 'IPR013785'}
{'taxon': '196627'}
{'InterPro': 'IPR011040', 'UniProtKB/TrEMBL': 'Q8NMC8'}
{'taxon': '196627'}
{'GOA': 'Q6M2J2', 'InterPro': 'IPR011991', 'UniProtKB/TrEMBL': 'Q6M2J2'}
{'taxon': '196627'}
{'GOA': 'Q6M5M9', 'InterPro': 'IPR027417', 'UniProtKB/TrEMBL': 'Q6M5M9'}
{'taxon': '196627'}
{'GOA': 'Q8NQU3', 'InterPro': 'IPR010065', 'UniProtKB/TrEMBL': 'Q8NQU3'}
{'taxon': '196627'}
{'GOA': 'Q6M5M7', 'InterPro': 'IPR001638', 'UniProtKB/TrEMBL': 'Q6M5M7'}
{'taxon': '196627'}
{'InterPro': 'IPR009677', 'UniProtKB/TrEMBL': 'Q8NQU1'}
{'taxon': '196627'}
{'InterPro': 'IPR007163', 'UniProtKB/TrEMBL': 'Q8NQU0'}
{'taxon': '196627'}
{'GOA': 'Q6M5M4', 'InterPro': 'IPR0137

In [6]:
gene_mapper = {}

for gene in model.genes:
    if 'kegg.genes' in gene.annotation:
        gene_mapper[gene.id] = gene.annotation['kegg.genes'][4:]

In [7]:
#replace the ncbi gene names in the GPR relation
def replace_ids(gpr, mapper):
    if pd.isna(gpr):  # Handle missing values
        return gpr
    for old_id, new_id in mapper.items():
        gpr = gpr.replace(old_id, new_id)
    return gpr

id_mapper_icg_gpr = id_mapper_icg.copy()
# Apply the replacement function to the GPR column
id_mapper_icg_gpr["GPR"] = id_mapper_icg_gpr["GPR"].apply(lambda x: replace_ids(x, gene_mapper))
id_mapper_icg_gpr

,rxn_id,kegg_id,rhea_id,mnx_id,Reactants,Products,EC,GPR
0,2AGPEAT160,NaN,NaN,MNXR94760,[],[],2.3.1.40,
1,2AGPEAT180,NaN,NaN,MNXR94762,[],[],2.3.1.40,
2,2MAHMP,NaN,NaN,MNXR142616,"[[C00001, C01328, C00001;C01328;D00001;D03703;...","[C04556, C00080, [C00009, C00009;D05467]]",NaN,lcl_NC_006958_1_prot_WP_003855288_1_2949 or cg...
3,3HAD140,R04568,NaN,MNXR94879,[C04688],"[[C00001, C01328, C00001;C01328;D00001;D03703;...","[4.2.1.61, 2.3.1.86, 2.3.1.-, 2.3.1.85, 4.2.1.59]",
4,3MBt2pp,NaN,NaN,MNXR94916,"[C00080, C08262]","[C08262, C00080]",NaN,cg0953
...,...,...,...,...,...,...,...,...
1142,Ca2t2,NaN,NaN,NaN,"[C00076, C00080]","[C00076, C00080]",NaN,WP_011014107_1
1143,CAT,R00009,"[20309, 20310, 20311, 20312]",MNXR96455,"[[C00027, C00027;D00008]]","[[C00001, C01328, C00001;C01328;D00001;D03703;...","[1.11.1.21, 1.11.1.6]",WP_011013509_1
1144,SUCDi,NaN,"[29187, 29188, 29189, 29190]",MNXR99641,[],[],NaN,WP_011013606_1 and WP_011013607_1
1145,PRFGS_1,NaN,"[17129, 17131, 17130, 17132]",MNXR138721,"[[C00002, C00002;D08646], C04376, [C00064, C00...","[[C00008, C00008;G11113], C04640, [C00025, C00...",6.3.5.3,cg2865


## 2. Get all the protein information

Besides information about the reactions, we need the relation to the proteins in order to establish a protein allocation model. We do this using the gene-protein-reaction association obtained from the BIGG model. The genes is the model can be mapped to the Uniprot accessions, which can be mapped to the KEGG reactions using the Rhea database.

### 2.1 Parse the BIGG gene-protein-reaction associations

In [8]:
#You need to adjust this to find the geneid/locustag for your microbe
#locustag_regex =r'\b([b|s]\d{4})\b'
def extract_b_numbers(text):
    if pd.isna(text):
        return text
    return re.findall(locustag_regex, text)
id_mapper_df = id_mapper_icg_gpr.copy()

id_mapper_df['b_number'] = id_mapper_df['GPR'].apply(extract_b_numbers)
id_mapper_df = id_mapper_df.explode('b_number', ignore_index=True)
id_mapper_df = id_mapper_df.explode('EC', ignore_index=True)
id_mapper_df

,rxn_id,kegg_id,rhea_id,mnx_id,Reactants,Products,EC,GPR,b_number
0,2AGPEAT160,NaN,NaN,MNXR94760,[],[],2.3.1.40,,NaN
1,2AGPEAT180,NaN,NaN,MNXR94762,[],[],2.3.1.40,,NaN
2,2MAHMP,NaN,NaN,MNXR142616,"[[C00001, C01328, C00001;C01328;D00001;D03703;...","[C04556, C00080, [C00009, C00009;D05467]]",NaN,lcl_NC_006958_1_prot_WP_003855288_1_2949 or cg...,lcl_NC_006958_1_prot_WP_003855288_1_2949
3,2MAHMP,NaN,NaN,MNXR142616,"[[C00001, C01328, C00001;C01328;D00001;D03703;...","[C04556, C00080, [C00009, C00009;D05467]]",NaN,lcl_NC_006958_1_prot_WP_003855288_1_2949 or cg...,cg3199
4,3HAD140,R04568,NaN,MNXR94879,[C04688],"[[C00001, C01328, C00001;C01328;D00001;D03703;...",4.2.1.61,,NaN
...,...,...,...,...,...,...,...,...,...
2449,CAT,R00009,"[20309, 20310, 20311, 20312]",MNXR96455,"[[C00027, C00027;D00008]]","[[C00001, C01328, C00001;C01328;D00001;D03703;...",1.11.1.6,WP_011013509_1,WP_011013509_1
2450,SUCDi,NaN,"[29187, 29188, 29189, 29190]",MNXR99641,[],[],NaN,WP_011013606_1 and WP_011013607_1,WP_011013606_1
2451,SUCDi,NaN,"[29187, 29188, 29189, 29190]",MNXR99641,[],[],NaN,WP_011013606_1 and WP_011013607_1,WP_011013607_1
2452,PRFGS_1,NaN,"[17129, 17131, 17130, 17132]",MNXR138721,"[[C00002, C00002;D08646], C04376, [C00064, C00...","[[C00008, C00008;G11113], C04640, [C00025, C00...",6.3.5.3,cg2865,cg2865


### 2.2 Parse the Uniprot data for merging

In [9]:
def extract_kegg_gene_ids(text):
    if pd.isna(text):
        return text
    return re.findall(r'Cgl\d{4}', text)

# load the uniprot information
uniprot_df = pd.read_excel(UNIPROT_FILE_PATH)
#get the gene id from the gene names
uniprot_df['b_number'] = uniprot_df['Gene Names'].apply(extract_b_numbers)
# uniprot_df['kegg_gene'] = uniprot_df['Gene Names'].apply(extract_kegg_gene_ids)
uniprot_df = uniprot_df.explode('b_number', ignore_index=True)
# uniprot_df = uniprot_df.explode('kegg_gene', ignore_index=True)

uniprot_df = uniprot_df.drop(['Gene Names'], axis=1)
uniprot_df_genemapper = uniprot_df.dropna(subset='b_number')[['Entry', 'b_number', 'Mass', 'Length']]
uniprot_df_ecmapper = uniprot_df.dropna(subset='EC number')[['Entry', 'EC number', 'Mass', 'Length', 'b_number']]


### 2.3 Match Uniprot to BIGG data

In [10]:
# Merge first on b_number
id_mapper_with_protein = pd.merge(
    id_mapper_df, 
    uniprot_df_genemapper, 
    on='b_number', 
    how='left'
)

# Merge on EC number for rows where b_number-based merge failed
fallback_merge = pd.merge(
    id_mapper_df, 
    uniprot_df_ecmapper, 
    left_on='EC', 
    right_on='EC number', 
    how='left'
)

# Combine the two merges: prefer the first merge, fill missing values from the second
id_mapper_with_protein['Entry'] = id_mapper_with_protein['Entry'].fillna(fallback_merge['Entry'])
id_mapper_with_protein

,rxn_id,kegg_id,rhea_id,mnx_id,Reactants,Products,EC,GPR,b_number,Entry,Mass,Length
0,2AGPEAT160,NaN,NaN,MNXR94760,[],[],2.3.1.40,,NaN,NaN,NaN,NaN
1,2AGPEAT180,NaN,NaN,MNXR94762,[],[],2.3.1.40,,NaN,NaN,NaN,NaN
2,2MAHMP,NaN,NaN,MNXR142616,"[[C00001, C01328, C00001;C01328;D00001;D03703;...","[C04556, C00080, [C00009, C00009;D05467]]",NaN,lcl_NC_006958_1_prot_WP_003855288_1_2949 or cg...,lcl_NC_006958_1_prot_WP_003855288_1_2949,NaN,NaN,NaN
3,2MAHMP,NaN,NaN,MNXR142616,"[[C00001, C01328, C00001;C01328;D00001;D03703;...","[C04556, C00080, [C00009, C00009;D05467]]",NaN,lcl_NC_006958_1_prot_WP_003855288_1_2949 or cg...,cg3199,NaN,NaN,NaN
4,3HAD140,R04568,NaN,MNXR94879,[C04688],"[[C00001, C01328, C00001;C01328;D00001;D03703;...",4.2.1.61,,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
2450,CAT,R00009,"[20309, 20310, 20311, 20312]",MNXR96455,"[[C00027, C00027;D00008]]","[[C00001, C01328, C00001;C01328;D00001;D03703;...",1.11.1.6,WP_011013509_1,WP_011013509_1,Q8NLR8,NaN,NaN
2451,SUCDi,NaN,"[29187, 29188, 29189, 29190]",MNXR99641,[],[],NaN,WP_011013606_1 and WP_011013607_1,WP_011013606_1,NaN,NaN,NaN
2452,SUCDi,NaN,"[29187, 29188, 29189, 29190]",MNXR99641,[],[],NaN,WP_011013606_1 and WP_011013607_1,WP_011013607_1,NaN,NaN,NaN
2453,PRFGS_1,NaN,"[17129, 17131, 17130, 17132]",MNXR138721,"[[C00002, C00002;D08646], C04376, [C00064, C00...","[[C00008, C00008;G11113], C04640, [C00025, C00...",6.3.5.3,cg2865,cg2865,Q8NQV9,NaN,NaN


## 3. Match the BiGG model reaction and enzymes to the kcats from GotEnzymes
Find first all enzymes for a reaction, then for each enzyme determine if the kcats from GotEnzymes are for a forward or reverse reaction

In [11]:
id_mapper_final = id_mapper_with_protein
id_mapper_final = id_mapper_final.rename({'Entry':'enzyme_id'}, axis=1)
id_mapper_final = id_mapper_final.explode('kegg_id', ignore_index=True)

# id_mapper_final = id_mapper_final.drop_duplicates(subset = ['kegg_id'])
id_mapper_final.head()

,rxn_id,kegg_id,rhea_id,mnx_id,Reactants,Products,EC,GPR,b_number,enzyme_id,Mass,Length
0,2AGPEAT160,NaN,NaN,MNXR94760,[],[],2.3.1.40,,NaN,NaN,NaN,NaN
1,2AGPEAT180,NaN,NaN,MNXR94762,[],[],2.3.1.40,,NaN,NaN,NaN,NaN
2,2MAHMP,NaN,NaN,MNXR142616,"[[C00001, C01328, C00001;C01328;D00001;D03703;...","[C04556, C00080, [C00009, C00009;D05467]]",NaN,lcl_NC_006958_1_prot_WP_003855288_1_2949 or cg...,lcl_NC_006958_1_prot_WP_003855288_1_2949,NaN,NaN,NaN
3,2MAHMP,NaN,NaN,MNXR142616,"[[C00001, C01328, C00001;C01328;D00001;D03703;...","[C04556, C00080, [C00009, C00009;D05467]]",NaN,lcl_NC_006958_1_prot_WP_003855288_1_2949 or cg...,cg3199,NaN,NaN,NaN
4,3HAD140,R04568,NaN,MNXR94879,[C04688],"[[C00001, C01328, C00001;C01328;D00001;D03703;...",4.2.1.61,,NaN,NaN,NaN,NaN


In [12]:
#get the json file from GotEnzymes and parse into workable format
#read in the json file obtained from gotEnzymes (for 'eco')
eco_enzymes_json = pd.read_json(GOTENZYMES_FILE_PATH)
#convert to a readible format
eco_enzymes = eco_enzymes_json.enzymes.to_list()
eco_enzymes_df = pd.DataFrame(eco_enzymes)

#ensure there is only one ec number per row
eco_enzymes_df['ec_number'] = eco_enzymes_df['ec_number'].str.split(pat=';')
eco_enzymes_df = eco_enzymes_df.explode('ec_number', ignore_index=True)
eco_enzymes_df = eco_enzymes_df.dropna(axis=0, subset=['ec_number'])
eco_enzymes_df = eco_enzymes_df.drop(['organism', 'domain'], axis = 1)
eco_enzymes_df.head()

,gene,reaction_id,ec_number,compound,kcat_values
0,Cgl0023,R01664,3.1.3.5,C00239,5.4287
1,Cgl0023,R01664,3.1.3.89,C00239,5.4287
2,Cgl0023,R01664,3.1.3.-,C00239,5.4287
3,Cgl0023,R00511,3.1.3.5,C00475,5.9248
4,Cgl0023,R00511,3.1.3.91,C00475,5.9248


In [21]:
# match BiGG and GotEnzymes based on gene id
eco_enzymes_merged = id_mapper_final.merge(eco_enzymes_df, 
                              left_on =['b_number', 'kegg_id'], 
                              right_on = ['gene', 'reaction_id'],
                             how = 'left').drop(['gene', 'reaction_id'], axis=1).rename({'b_number':'gene'}, axis=1)
eco_enzymes_merged

,rxn_id,kegg_id,rhea_id,mnx_id,Reactants,Products,EC,GPR,gene,enzyme_id,Mass,Length,ec_number,compound,kcat_values
0,2AGPEAT160,NaN,NaN,MNXR94760,[],[],2.3.1.40,,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2AGPEAT180,NaN,NaN,MNXR94762,[],[],2.3.1.40,,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2MAHMP,NaN,NaN,MNXR142616,"[[C00001, C01328, C00001;C01328;D00001;D03703;...","[C04556, C00080, [C00009, C00009;D05467]]",NaN,lcl_NC_006958_1_prot_WP_003855288_1_2949 or cg...,lcl_NC_006958_1_prot_WP_003855288_1_2949,NaN,NaN,NaN,NaN,NaN,NaN
3,2MAHMP,NaN,NaN,MNXR142616,"[[C00001, C01328, C00001;C01328;D00001;D03703;...","[C04556, C00080, [C00009, C00009;D05467]]",NaN,lcl_NC_006958_1_prot_WP_003855288_1_2949 or cg...,cg3199,NaN,NaN,NaN,NaN,NaN,NaN
4,3HAD140,R04568,NaN,MNXR94879,[C04688],"[[C00001, C01328, C00001;C01328;D00001;D03703;...",4.2.1.61,,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2480,CAT,R00009,"[20309, 20310, 20311, 20312]",MNXR96455,"[[C00027, C00027;D00008]]","[[C00001, C01328, C00001;C01328;D00001;D03703;...",1.11.1.6,WP_011013509_1,WP_011013509_1,Q8NLR8,NaN,NaN,NaN,NaN,NaN
2481,SUCDi,NaN,"[29187, 29188, 29189, 29190]",MNXR99641,[],[],NaN,WP_011013606_1 and WP_011013607_1,WP_011013606_1,NaN,NaN,NaN,NaN,NaN,NaN
2482,SUCDi,NaN,"[29187, 29188, 29189, 29190]",MNXR99641,[],[],NaN,WP_011013606_1 and WP_011013607_1,WP_011013607_1,NaN,NaN,NaN,NaN,NaN,NaN
2483,PRFGS_1,NaN,"[17129, 17131, 17130, 17132]",MNXR138721,"[[C00002, C00002;D08646], C04376, [C00064, C00...","[[C00008, C00008;G11113], C04640, [C00025, C00...",6.3.5.3,cg2865,cg2865,Q8NQV9,NaN,NaN,NaN,NaN,NaN


### 6. Extract directionalities and gapfill
Reactions which are associated with an enzyme, but not with a kcat from GotEnzymes will be assignes
a default kcat and if required a default molmass.

In [22]:
# Get directionalities and fill in the gaps
eco_enzymes_merged['direction'] = 'f'
                                              
for index, row in eco_enzymes_merged.iterrows():
    #there is a kegg id and kcat associated to this reaction
    if not isinstance(row.kcat_values, float):
        if row.compound in row.Products:
            eco_enzymes_merged.direction.iloc[index] = 'b'
        elif not row.compound in row.Reactants:
            print(f'Compound {row["compound"]} is not associated to reaction {row["kegg_id"]}')
            continue
    #there is no kcat associated to this reaction, try to use EC number to get a kcat value
    else: 
        got_enzyme_info = eco_enzymes_df[eco_enzymes_df['ec_number']==row.EC]
        for i, info in got_enzyme_info.iterrows():
            #is there any enzyme association if we look at the metabolites in the reaction and the EC number?
            if info.compound in row.Products:
                direction = 'b'
            elif info.compound in row.Reactants:
                direction = 'f'
            else:
                continue
            #change the kcat if it is not there already or create a new enzyme association
            if np.isnan(row.kcat_values):
                eco_enzymes_merged.direction.iloc[index] = 'b'
                eco_enzymes_merged.kcat_values.iloc[index] = info.kcat_values
            else:
                eco_enzymes_merged.loc[len(eco_enzymes_merged)] = row.to_list()[:-2]+[info.kcat_values, 'b']
        #add default information for unmappable proteins
        
        if (row.GPR == '') and not isinstance(row.EC, float): 
            eco_enzymes_merged.gene[index] = [f'Gene_{row.rxn_id}_{row.EC}']
            eco_enzymes_merged.GPR[index] = f'Gene_{row.rxn_id}_{row.EC}'
            eco_enzymes_merged.enzyme_id[index] = f'Enzyme_{row.rxn_id}_{row.EC}'
        if isinstance(row.gene, float) and len(row.GPR)>0:
            eco_enzymes_merged.gene[index] = [row.GPR]
        if not isinstance(row.enzyme_id, str) and len(row.GPR)>0:
            eco_enzymes_merged.enzyme_id[index] = f'Enzyme_{row.gene}'
        
        if np.isnan(row.kcat_values) & (isinstance(row.enzyme_id, str)) & (row.GPR!= 's0001'):
            eco_enzymes_merged.direction.iloc[index] = 'f'
            eco_enzymes_merged.kcat_values.iloc[index] = DEFAULT_KCAT
        if np.isnan(row.Mass) & (len(row.GPR)>2) & (row.GPR!= 's0001'):
            eco_enzymes_merged.Mass.iloc[index] = DEFAULT_PROT_MW
        
        
eco_enzymes_mapped = eco_enzymes_merged.drop(['ec_number', 'compound'], axis=1).rename({'Mass':'molMass',
                                                                                       'b_number': 'gene'}, axis =1)
eco_enzymes_mapped

/tmp/ipykernel_689443/253922216.py:32: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  eco_enzymes_merged.gene[index] = [f'Gene_{row.rxn_id}_{row.EC}']
/tmp/ipykernel_689443/253922216.py:33: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  eco_enzymes_merged.GPR[index] = f'Gene_{row.rxn_id}_{row.EC}'
/tmp/ipykernel_689443/253922216.py:34: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  eco_enzymes_merged.enzyme_id[index] = f'Enzyme_{row.rxn_i

,rxn_id,kegg_id,rhea_id,mnx_id,Reactants,Products,EC,GPR,gene,enzyme_id,molMass,Length,kcat_values,direction
0,2AGPEAT160,NaN,NaN,MNXR94760,[],[],2.3.1.40,Gene_2AGPEAT160_2.3.1.40,[Gene_2AGPEAT160_2.3.1.40],Enzyme_2AGPEAT160_2.3.1.40,NaN,NaN,NaN,f
1,2AGPEAT180,NaN,NaN,MNXR94762,[],[],2.3.1.40,Gene_2AGPEAT180_2.3.1.40,[Gene_2AGPEAT180_2.3.1.40],Enzyme_2AGPEAT180_2.3.1.40,NaN,NaN,NaN,f
2,2MAHMP,NaN,NaN,MNXR142616,"[[C00001, C01328, C00001;C01328;D00001;D03703;...","[C04556, C00080, [C00009, C00009;D05467]]",NaN,lcl_NC_006958_1_prot_WP_003855288_1_2949 or cg...,lcl_NC_006958_1_prot_WP_003855288_1_2949,Enzyme_lcl_NC_006958_1_prot_WP_003855288_1_2949,39959.4825,NaN,NaN,f
3,2MAHMP,NaN,NaN,MNXR142616,"[[C00001, C01328, C00001;C01328;D00001;D03703;...","[C04556, C00080, [C00009, C00009;D05467]]",NaN,lcl_NC_006958_1_prot_WP_003855288_1_2949 or cg...,cg3199,Enzyme_cg3199,39959.4825,NaN,NaN,f
4,3HAD140,R04568,NaN,MNXR94879,[C04688],"[[C00001, C01328, C00001;C01328;D00001;D03703;...",4.2.1.61,Gene_3HAD140_4.2.1.61,[Gene_3HAD140_4.2.1.61],Enzyme_3HAD140_4.2.1.61,NaN,NaN,NaN,f
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2480,CAT,R00009,"[20309, 20310, 20311, 20312]",MNXR96455,"[[C00027, C00027;D00008]]","[[C00001, C01328, C00001;C01328;D00001;D03703;...",1.11.1.6,WP_011013509_1,WP_011013509_1,Q8NLR8,39959.4825,NaN,13.7,f
2481,SUCDi,NaN,"[29187, 29188, 29189, 29190]",MNXR99641,[],[],NaN,WP_011013606_1 and WP_011013607_1,WP_011013606_1,Enzyme_WP_011013606_1,39959.4825,NaN,NaN,f
2482,SUCDi,NaN,"[29187, 29188, 29189, 29190]",MNXR99641,[],[],NaN,WP_011013606_1 and WP_011013607_1,WP_011013607_1,Enzyme_WP_011013607_1,39959.4825,NaN,NaN,f
2483,PRFGS_1,NaN,"[17129, 17131, 17130, 17132]",MNXR138721,"[[C00002, C00002;D08646], C04376, [C00064, C00...","[[C00008, C00008;G11113], C04640, [C00025, C00...",6.3.5.3,cg2865,cg2865,Q8NQV9,39959.4825,NaN,13.7,f


In [23]:
eco_enzymes_mapped.gene = [[g] for g in eco_enzymes_mapped.gene]
protein2gene, gene2protein = get_protein_gene_mapping(eco_enzymes_mapped, model)


# Ensure the enzyme complexes are merged on one row
eco_enzymes_mapped = merge_enzyme_complexes(eco_enzymes_mapped, gene2protein)

# Save the adjusted dataframe or continue processing
eco_enzymes_mapped

,rxn_id,kegg_id,rhea_id,mnx_id,Reactants,Products,EC,GPR,gene,enzyme_id,molMass,Length,kcat_values,direction
2,2MAHMP,NaN,NaN,MNXR142616,"[[C00001, C01328, C00001;C01328;D00001;D03703;...","[C04556, C00080, [C00009, C00009;D05467]]",NaN,lcl_NC_006958_1_prot_WP_003855288_1_2949 or cg...,[lcl_NC_006958_1_prot_WP_003855288_1_2949],Enzyme_lcl_NC_006958_1_prot_WP_003855288_1_2949,39959.4825,NaN,NaN,f
3,2MAHMP,NaN,NaN,MNXR142616,"[[C00001, C01328, C00001;C01328;D00001;D03703;...","[C04556, C00080, [C00009, C00009;D05467]]",NaN,lcl_NC_006958_1_prot_WP_003855288_1_2949 or cg...,[cg3199],Enzyme_cg3199,39959.4825,NaN,NaN,f
9,3MBt2pp,NaN,NaN,MNXR94916,"[C00080, C08262]","[C08262, C00080]",NaN,cg0953,[cg0953],Q8NMS0,39959.4825,NaN,13.7,f
10,3MBt4pp,NaN,NaN,MNXR94917,"[C01330, C08262]","[C01330, C08262]",NaN,cg0953,[cg0953],Q8NS46,39959.4825,NaN,13.7,f
12,3OXCOAT,R00829,"[19482, 19483, 19481, 19484]",MNXR94967,"[C02232, C00010]","[C00091, C00024]",2.3.1.174,cg2625,[cg2625],Enzyme_cg2625,39959.4825,NaN,NaN,f
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2439,ZNabcpp,NaN,"[29795, 29796, 29797, 29798]",MNXR105281,"[[C00002, C00002;D08646], [C00001, C01328, C00...","[C00038, [C00008, C00008;G11113], C00080, [C00...",3.6.3.5,cg1040 or cg2912 or (cg0042 and cg0043) or (cg...,[cg2912],Enzyme_cg2912,39959.4825,NaN,NaN,f
2440,ZNabcpp,NaN,"[29795, 29796, 29797, 29798]",MNXR105281,"[[C00002, C00002;D08646], [C00001, C01328, C00...","[C00038, [C00008, C00008;G11113], C00080, [C00...",3.6.3.5,cg1040 or cg2912 or (cg0042 and cg0043) or (cg...,"[cg0042, cg0043]",Enzyme_cg0042,39959.4825,NaN,NaN,f
2441,ZNabcpp,NaN,"[29795, 29796, 29797, 29798]",MNXR105281,"[[C00002, C00002;D08646], [C00001, C01328, C00...","[C00038, [C00008, C00008;G11113], C00080, [C00...",3.6.3.5,cg1040 or cg2912 or (cg0042 and cg0043) or (cg...,"[cg0042, cg0043]",Enzyme_cg0043,39959.4825,NaN,NaN,f
2442,ZNabcpp,NaN,"[29795, 29796, 29797, 29798]",MNXR105281,"[[C00002, C00002;D08646], [C00001, C01328, C00...","[C00038, [C00008, C00008;G11113], C00080, [C00...",3.6.3.5,cg1040 or cg2912 or (cg0042 and cg0043) or (cg...,"[cg0042, cg0043]",Enzyme_cg0042,39959.4825,NaN,NaN,f


In [24]:
#save the dataframe
eco_enzymes_mapped = eco_enzymes_mapped.drop_duplicates(subset = ['rxn_id', 'enzyme_id', 'direction'],
                                                       ignore_index=True)
eco_enzymes_mapped.to_excel(AE_OUTPUT_FILE_PATH)

### 7. Save the dataframe to the proper format for building PAMs

In [25]:
# Get the information about the enzyme sectors
unused_enzymes_df = pd.read_excel(PAM_DATA_FILE_PATH,
                                 sheet_name = 'ExcessEnzymes')
translational_protein_df = pd.read_excel(PAM_DATA_FILE_PATH,
                                 sheet_name = 'Translational')

In [26]:
# Save it to a new excel file
with pd.ExcelWriter(OUTPUT_FILE_PATH) as writer:
    eco_enzymes_mapped.to_excel(writer, sheet_name='ActiveEnzymes')
    translational_protein_df.to_excel(writer, sheet_name='Translational')
    unused_enzymes_df.to_excel(writer, sheet_name = 'UnusedEnzyme')